# Code initializaction

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 120)

# %matplotlib inline

## Data loading

In [2]:
import glob

parquet_files = glob.glob('data/*.parquet')

df_list = []
for file in parquet_files:
    df = pd.read_parquet(file, engine='fastparquet')
    df_list.append(df)

if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
else:
    print("No se encontraron archivos .parquet en el directorio 'sample_data'.")


combined_df.describe()
combined_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 223549 entries, 0 to 223548
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             223549 non-null  object
 1   comment_text   223549 non-null  object
 2   toxic          223549 non-null  int64 
 3   severe_toxic   223549 non-null  int64 
 4   obscene        223549 non-null  int64 
 5   threat         223549 non-null  int64 
 6   insult         223549 non-null  int64 
 7   identity_hate  223549 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 13.6+ MB


## Cleaning and simplification of columns

In [3]:
# ID is not an relevant columns thats really describe some behavior of data.
relevant_columns = combined_df.columns.drop(['id'])
toxic_df = combined_df[relevant_columns]

# Some study
Esta data al ser enfocada a comentarios y clasificada por tipo de comentario. Asumimos que ninguna columna tiene valores `NaN`, incluso la misma columna principal `coment_text`.

In [4]:
toxic_df.shape, toxic_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 223549 entries, 0 to 223548
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   comment_text   223549 non-null  object
 1   toxic          223549 non-null  int64 
 2   severe_toxic   223549 non-null  int64 
 3   obscene        223549 non-null  int64 
 4   threat         223549 non-null  int64 
 5   insult         223549 non-null  int64 
 6   identity_hate  223549 non-null  int64 
dtypes: int64(6), object(1)
memory usage: 11.9+ MB


((223549, 7), None)

## Análisis de las variables booleanas

In [5]:
# Demostrando que ninguna columna tiene valores NaN
possible_NaN_in_data = np.array([
    toxic_df[column].isna().astype(int)
    for column in toxic_df.columns])
print(possible_NaN_in_data.sum())

total_rows = toxic_df.shape

0


Las variables booleanas representan la clasificación de los comentarios.
En esta parte, observamos la gran mayoría de los comentarios en este dataset, no están clasificados.

In [6]:
bool_columns = toxic_df.select_dtypes(include=['number']).columns
clean_clasification_mask = toxic_df[bool_columns].sum(axis=1) == 0
non_clasified_qyt = clean_clasification_mask.mean() * 100
print(f'Non Clisified rows: {non_clasified_qyt:.2f}%, arrond {clean_clasification_mask.sum()} registers')
print(f'Clisified rows: {(100-non_clasified_qyt):.2f}%, arrond {(~clean_clasification_mask).sum()} registers')


Non Clisified rows: 89.95%, arrond 201081 registers
Clisified rows: 10.05%, arrond 22468 registers


Con esta parte, detallamos un poco la proporción los registros clasificados, prestando atención en la columna de `toxic`. Se puede observar que existen clasificaciones cuando `toxic` está desactivado.

In [7]:
resumen = toxic_df.groupby('toxic')[list(bool_columns.drop('toxic'))].agg(['sum', 'mean'])
resumen_pct = resumen.xs('mean', axis=1, level=1) * 100
resumen_qty = resumen.xs('sum', axis=1, level=1)
print(resumen_pct)
print(resumen_qty)


       severe_toxic    obscene    threat     insult  identity_hate
toxic                                                             
0          0.000000   0.290852  0.017313   0.305691       0.060347
1          9.175084  54.021698  3.058361  49.971942       9.329405
       severe_toxic  obscene  threat  insult  identity_hate
toxic                                                      
0                 0      588      35     618            122
1              1962    11552     654   10686           1995


In [8]:
threat_mask = (toxic_df['threat'] == 1) & (toxic_df['toxic'] == 0)
threat_not_toxic_rows = toxic_df[threat_mask]
comments = threat_not_toxic_rows['comment_text']

# Definimos el nombre del archivo
file_name = "comentarios_amenaza_no_toxicos.txt"

# Abrimos el archivo en modo escritura ('w') con codificación utf-8 por los caracteres especiales
with open(file_name, "w", encoding="utf-8") as f:
    for i, comment in enumerate(comments, 1):
        f.write(f"--- Comentario {i} ---\n")
        f.write(f"{comment}\n\n")

print(f"¡Listo! Se han guardado {len(comments)} comentarios en {file_name}")

¡Listo! Se han guardado 35 comentarios en comentarios_amenaza_no_toxicos.txt


**Conclusion**: Our problem consists in determinate how coment is toxic, so, all those bools columns are unlesses.
So, we clear those columns in the final df and we have to take more attention in `coment_text` and explore his statistics fatures.

## `coment_text` column analitics

In [9]:
# Generating new coment_text-only dataframe
comment_text_df = toxic_df[['comment_text', 'toxic']]

In [ ]:
import re
from collections import Counter

def top_n_words(text_series, n=10):
    cnt = Counter()
    for txt in text_series.dropna():
        tokens = [t for t in re.findall(r'\w+', txt.lower(), flags=re.UNICODE) if t.isalpha()]
        cnt.update(tokens)
    most = cnt.most_common(n)
    total_docs = len(text_series.dropna())
    return [(w, round(c / total_docs, 2), c) for w, c in most]

top_toxic = top_n_words(comment_text_df.loc[comment_text_df['toxic'] == 1, 'comment_text'], 10)
top_not_toxic = top_n_words(comment_text_df.loc[comment_text_df['toxic'] == 0, 'comment_text'], 10)

print("Toxic comments:", top_toxic)
print("Non-toxic comments:", top_not_toxic)

Toxic comments: [('you', 49474, 2, 2), ('i', 29562, 1, 2), ('the', 27625, 1, 2), ('a', 27053, 1, 2), ('and', 20702, 1, 2), ('to', 20166, 1, 2), ('is', 17751, 1, 2), ('fuck', 15950, 1, 2), ('of', 14634, 1, 2), ('your', 10968, 1, 2)]
Non-toxic comments: [('the', 657528, 3, 2), ('to', 385128, 2, 2), ('i', 294543, 1, 2), ('of', 292551, 1, 2), ('and', 284570, 1, 2), ('a', 272589, 1, 2), ('you', 242107, 1, 2), ('is', 227008, 1, 2), ('that', 210002, 1, 2), ('it', 194794, 1, 2)]


In [ ]:
import matplotlib.pyplot as plt

# Extraer palabras y conteos desde las listas ya calculadas
words_t, counts_t = zip(*[(w, c) for w, c, _ in top_toxic])
words_nt, counts_nt = zip(*[(w, c) for w, c, _ in top_not_toxic])

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# Tóxicos
axes[0].barh(words_t[::-1], counts_t[::-1], color='darkred')
axes[0].set_title('Top palabras - Tóxicos')
axes[0].set_xlabel('Frecuencia (conteo absoluto)')
for i, v in enumerate(counts_t[::-1]):
    axes[0].text(v, i, f' {v}', va='center')

# No tóxicos
axes[1].barh(words_nt[::-1], counts_nt[::-1], color='steelblue')
axes[1].set_title('Top palabras - No tóxicos')
axes[1].set_xlabel('Frecuencia (conteo absoluto)')
for i, v in enumerate(counts_nt[::-1]):
    axes[1].text(v, i, f' {v}', va='center')

plt.tight_layout()
plt.show()

ValueError: too many values to unpack (expected 3)